## Exploration Notebook
The idea of this notebook is to have a clear exploration of different aspects of the ACLED data for Myanmar specifcaly 


#### Setup 

In [2]:
import pandas as pd

#### Basic analysis

In [7]:
df = pd.read_csv('../data/raw/acled_myanmar_2026-04-22.csv')

#### Analysis of Sub_Type_Event

In [11]:
print(df.sub_event_type.value_counts())

# Classification of sub_event_type into ACLED's official parent event types:
# Battles, Violence against civilians, Explosions/Remote violence, 
# Strategic developments, Protests, Riots
def classify_sub_event_type(sub_event_type):
    battles = [
        'Armed clash',
        'Government regains territory',
        'Non-state actor overtakes territory'
    ]
    
    violence_against_civilians = [
        'Attack',
        'Abduction/forced disappearance',
        'Sexual violence'
    ]
    
    remote_violence = [
        'Air/drone strike',
        'Shelling/artillery/missile attack',
        'Remote explosive/landmine/IED',
        'Grenade',
        'Suicide bomb',
        'Chemical weapon'
    ]
    
    strategic_developments = [
        'Agreement',
        'Arrests',
        'Change to group/activity',
        'Disrupted weapons use',
        'Headquarters or base established',
        'Non-violent transfer of territory',
        'Looting/property destruction',
        'Other'
    ]
    
    protests = [
        'Peaceful protest',
        'Protest with intervention',
        'Excessive force against protesters'
    ]
    
    riots = [
        'Violent demonstration',
        'Mob violence'
    ]
    
    if sub_event_type in battles:
        return 'Battles'
    elif sub_event_type in violence_against_civilians:
        return 'Violence against civilians'
    elif sub_event_type in remote_violence:
        return 'Explosions/Remote violence'
    elif sub_event_type in strategic_developments:
        return 'Strategic developments'
    elif sub_event_type in protests:
        return 'Protests'
    elif sub_event_type in riots:
        return 'Riots'
    else:
        return 'Unknown'

df['event_category'] = df['sub_event_type'].apply(classify_sub_event_type)
print(df['event_category'].value_counts())

# Sanity check — make sure nothing fell into 'Unknown'
print("\nUnclassified:", df[df['event_category'] == 'Unknown']['sub_event_type'].unique())

# Optional: territorial control signal flag (high-value events for your control mapping)
control_signals = [
    'Non-state actor overtakes territory',
    'Government regains territory',
    'Non-violent transfer of territory',
    'Headquarters or base established'
]
df['is_control_signal'] = df['sub_event_type'].isin(control_signals)
print(f"\nTerritorial control signal events: {df['is_control_signal'].sum()}")

sub_event_type
Armed clash                            27115
Peaceful protest                       17223
Air/drone strike                        9721
Attack                                  8950
Remote explosive/landmine/IED           8532
Shelling/artillery/missile attack       7923
Looting/property destruction            6733
Arrests                                 6335
Change to group/activity                3448
Abduction/forced disappearance          3125
Other                                   3105
Protest with intervention                645
Grenade                                  592
Disrupted weapons use                    523
Non-state actor overtakes territory      369
Sexual violence                          279
Excessive force against protesters       275
Mob violence                             164
Government regains territory             100
Non-violent transfer of territory         90
Violent demonstration                     83
Headquarters or base established        

### Analysing main actor per sub_type event 

In [20]:
# Quick reconnaissance — what do these columns look like?
print("Columns available:", [c for c in df.columns if 'actor' in c.lower() or 'inter' in c.lower()])
print("\nSample rows:")
print(df[['sub_event_type', 'actor1', 'actor2', 'interaction']].head(10))
print("\nActor1 unique count:", df['actor1'].nunique())
print("Interaction column dtype:", df['interaction'].dtype)
print("Interaction unique values:", sorted(df['interaction'].dropna().unique()))

Columns available: ['actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction']

Sample rows:
                      sub_event_type  \
0       Looting/property destruction   
1                        Armed clash   
2      Remote explosive/landmine/IED   
3       Looting/property destruction   
4                   Peaceful protest   
5                        Armed clash   
6                        Armed clash   
7                   Peaceful protest   
8                        Armed clash   
9  Non-violent transfer of territory   

                                              actor1  \
0              Unidentified Armed Group (Bangladesh)   
1  RCSS/SSA-S: Restoration Council of Shan State/...   
2                 Unidentified Armed Group (Myanmar)   
3                 Unidentified Armed Group (Myanmar)   
4                               Protesters (Myanmar)   
5  RCSS/SSA-S: Restoration Council of Shan State/...   
6  RCSS/SSA-S: Restoration Council of Shan S

In [22]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

target_subs = ['Armed clash', 'Air/drone strike', 'Arrests', 'Abduction/forced disappearance']
focus = df[df['sub_event_type'].isin(target_subs)].copy()

print(f"Total events: {len(focus):,}")
print(f"\nBreakdown by sub-event:")
print(focus['sub_event_type'].value_counts())

# Top 15 actor1 per sub-event type
for sub in target_subs:
    print(f"\n{'='*60}\nTop actor1 in '{sub}':")
    print(focus[focus['sub_event_type'] == sub]['actor1'].value_counts().head(15))

top_n = 10
top_actors = (
    focus.groupby(['sub_event_type', 'actor1'])
         .size()
         .reset_index(name='count')
         .sort_values(['sub_event_type', 'count'], ascending=[True, False])
         .groupby('sub_event_type')
         .head(top_n)
)

fig = px.bar(
    top_actors,
    x='count', y='actor1',
    color='sub_event_type',
    facet_col='sub_event_type', facet_col_wrap=2,
    orientation='h',
    height=800, width=1300,
    title=f'Top {top_n} actor1 by sub-event type'
)
fig.update_yaxes(matches=None, autorange='reversed')
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)
fig.show()


inter_labels = {
    1: 'State forces', 2: 'Rebel groups', 3: 'Political militias',
    4: 'Identity militias', 5: 'Rioters', 6: 'Protesters',
    7: 'Civilians', 8: 'External/Other'
}

# Decompose interaction code into the two actor type codes
# ACLED interaction is typically a two-digit code: tens digit = inter1, ones digit = inter2
focus['inter_str'] = focus['interaction'].astype(str).str.zfill(2)
focus['inter1_code'] = focus['inter_str'].str[0].astype(int)
focus['inter2_code'] = focus['inter_str'].str[1].astype(int)
focus['inter1_label'] = focus['inter1_code'].map(inter_labels)
focus['inter2_label'] = focus['inter2_code'].map(inter_labels).fillna('(none)')

# Build heatmap data per sub-event type
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=target_subs,
    horizontal_spacing=0.15, vertical_spacing=0.18
)

for i, sub in enumerate(target_subs):
    sub_df = focus[focus['sub_event_type'] == sub]
    pivot = (
        sub_df.groupby(['inter1_label', 'inter2_label'])
              .size()
              .unstack(fill_value=0)
    )
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns, y=pivot.index,
            colorscale='Blues',
            showscale=False,
            text=pivot.values, texttemplate='%{text}',
            hovertemplate='Actor1: %{y}<br>Actor2: %{x}<br>Count: %{z}<extra></extra>'
        ),
        row=row, col=col
    )

fig.update_layout(
    height=800, width=1300,
    title_text='Actor interaction heatmaps — who does what to whom'
)
fig.update_xaxes(tickangle=45)
fig.show()

Total events: 46,296

Breakdown by sub-event:
sub_event_type
Armed clash                       27115
Air/drone strike                   9721
Arrests                            6335
Abduction/forced disappearance     3125
Name: count, dtype: int64

Top actor1 in 'Armed clash':
actor1
Military Forces of Myanmar (2021-)                                                                 7941
Military Forces of Myanmar (2016-2021)                                                             2171
Military Forces of Myanmar (2011-2016)                                                             1506
KIO/KIA: Kachin Independence Organization/Kachin Independence Army                                 1264
KNU/KNLA: Karen National Union/Karen National Liberation Army                                      1083
ULA/AA: United League of Arakan/Arakan Army                                                         946
Unidentified Anti-Coup Armed Group                                                          

ValueError: invalid literal for int() with base 10: 'R'

### Can we make a analysis of the distribution of actor, for a few events sub_types? Non-state actor overtakes territory? 

In [15]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

df['event_date'] = pd.to_datetime(df['event_date'])

# ---------- Plot 1: one line per event_category over time ----------
monthly = (
    df.groupby([pd.Grouper(key='event_date', freq='ME'), 'event_category'])
      .size()
      .reset_index(name='count')
)

fig1 = px.line(
    monthly,
    x='event_date',
    y='count',
    color='event_category',
    title='ACLED events over time by category',
    labels={'event_date': 'Date', 'count': 'Number of events (monthly)', 'event_category': 'Category'}
)
fig1.update_layout(
    hovermode='x unified',
    width=1200,
    height=600,
    legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02)
)
fig1.show()


# ---------- Plot 2: territorial control signal events only ----------
monthly_control = (
    df[df['is_control_signal']]
      .groupby([pd.Grouper(key='event_date', freq='ME'), 'sub_event_type'])
      .size()
      .reset_index(name='count')
)

fig2 = px.line(
    monthly_control,
    x='event_date',
    y='count',
    color='sub_event_type',
    title='Territorial control signal events over time',
    labels={'event_date': 'Date', 'count': 'Number of events (monthly)', 'sub_event_type': 'Sub-event type'},
    markers=True
)
fig2.update_layout(
    hovermode='x unified',
    width=1200,
    height=550,
    legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02)
)
fig2.show()

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Make sure event_date is datetime
df['event_date'] = pd.to_datetime(df['event_date'])

# ---------- Aggregate monthly ----------
# Total fatalities per month (the background layer)
monthly_fatalities = (
    df.groupby(pd.Grouper(key='event_date', freq='ME'))['fatalities']
      .sum()
)

# Control-signal sub-events per month, broken out by sub_event_type (the foreground lines)
monthly_control = (
    df[df['is_control_signal']]
      .groupby([pd.Grouper(key='event_date', freq='ME'), 'sub_event_type'])
      .size()
      .unstack(fill_value=0)
)

# ---------- Plot with dual y-axes ----------
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Background: fatalities as translucent filled area
fig.add_trace(
    go.Scatter(
        x=monthly_fatalities.index,
        y=monthly_fatalities.values,
        name='Fatalities (total)',
        fill='tozeroy',
        line=dict(color='rgba(150, 150, 150, 0.4)', width=0),
        fillcolor='rgba(150, 150, 150, 0.25)',
        hovertemplate='%{x|%b %Y}<br>Fatalities: %{y:,}<extra></extra>'
    ),
    secondary_y=True
)

# Foreground: one line per control-signal sub_event_type
control_colors = {
    'Non-state actor overtakes territory': '#D85A30',
    'Government regains territory': '#185FA5',
    'Non-violent transfer of territory': '#1D9E75',
    'Headquarters or base established': '#7F77DD'
}

for sub_event in monthly_control.columns:
    fig.add_trace(
        go.Scatter(
            x=monthly_control.index,
            y=monthly_control[sub_event],
            name=sub_event,
            mode='lines+markers',
            line=dict(color=control_colors.get(sub_event, '#444'), width=2),
            marker=dict(size=5),
            hovertemplate='%{x|%b %Y}<br>' + sub_event + ': %{y}<extra></extra>'
        ),
        secondary_y=False
    )

fig.update_layout(
    title='Territorial control events vs total fatalities over time',
    hovermode='x unified',
    width=1300, height=600,
    legend=dict(orientation='h', y=-0.15),
    plot_bgcolor='white'
)
fig.update_yaxes(title_text='Control-signal events (count)', secondary_y=False, showgrid=True, gridcolor='rgba(0,0,0,0.05)')
fig.update_yaxes(title_text='Fatalities (monthly total)', secondary_y=True, showgrid=False)
fig.update_xaxes(title_text='Date', showgrid=False)

fig.show()

In [16]:
# Quick lagged correlation example
import pandas as pd

monthly = df.set_index('event_date').groupby([pd.Grouper(freq='ME'), 'event_category']).size().unstack(fill_value=0)
battles = monthly['Battles']
control = df[df['is_control_signal']].set_index('event_date').groupby(pd.Grouper(freq='ME')).size()
control = control.reindex(battles.index, fill_value=0)

for lag in range(0, 7):
    print(f"Lag {lag} months: corr = {battles.corr(control.shift(-lag)):.3f}")

    # Grid-based approach (simplest)
import numpy as np

df['lat_bin'] = (df['latitude'] // 0.5) * 0.5
df['lon_bin'] = (df['longitude'] // 0.5) * 0.5

cell_stats = df.groupby(['lat_bin', 'lon_bin']).agg(
    battles=('event_category', lambda x: (x == 'Battles').sum()),
    control_events=('is_control_signal', 'sum')
).reset_index()

# Correlation across cells
print(cell_stats[['battles', 'control_events']].corr())

# Logistic regression: do battles predict presence of control events in a cell?
from sklearn.linear_model import LogisticRegression
X = cell_stats[['battles']].values
y = (cell_stats['control_events'] > 0).astype(int).values
model = LogisticRegression().fit(X, y)
print(f"Coefficient: {model.coef_[0][0]:.4f}")

Lag 0 months: corr = 0.561
Lag 1 months: corr = 0.528
Lag 2 months: corr = 0.556
Lag 3 months: corr = 0.546
Lag 4 months: corr = 0.584
Lag 5 months: corr = 0.545
Lag 6 months: corr = 0.544
                battles  control_events
battles         1.00000         0.55901
control_events  0.55901         1.00000
Coefficient: 0.0088


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df['event_date'] = pd.to_datetime(df['event_date'])

# Build monthly time series per country
def build_monthly(df, country=None):
    sub = df if country is None else df[df['country'] == country]
    monthly = (
        sub.groupby([pd.Grouper(key='event_date', freq='ME'), 'event_category'])
           .size()
           .unstack(fill_value=0)
    )
    control = (
        sub[sub['is_control_signal']]
           .groupby(pd.Grouper(key='event_date', freq='ME'))
           .size()
    )
    monthly['Control_signals'] = control.reindex(monthly.index, fill_value=0)
    return monthly

# Pick one country to start — replace with your case
country = 'Myanmar'  # or 'Somalia', 'Nigeria', 'Ecuador'
monthly = build_monthly(df, country=country)
print(monthly.tail())




event_category  Battles  Explosions/Remote violence  Protests  Riots  \
event_date                                                             
2025-12-31          335                         445        52      1   
2026-01-31          324                         459        51      0   
2026-02-28          268                         369        31      0   
2026-03-31          335                         397        19      0   
2026-04-30          100                         175         6      0   

event_category  Strategic developments  Violence against civilians  \
event_date                                                           
2025-12-31                         257                         110   
2026-01-31                         337                         138   
2026-02-28                         286                         156   
2026-03-31                         293                         143   
2026-04-30                         112                          48   

even

In [18]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=monthly.index, y=monthly['Battles'],
               name='Battles', line=dict(color='steelblue', width=1.5)),
    secondary_y=False
)
fig.add_trace(
    go.Bar(x=monthly.index, y=monthly['Control_signals'],
           name='Control changes', marker_color='crimson', opacity=0.7),
    secondary_y=True
)

fig.update_layout(
    title=f'{country}: Battles vs territorial control events over time',
    hovermode='x unified',
    width=1200, height=500,
    legend=dict(orientation='h', y=1.1)
)
fig.update_yaxes(title_text='Battles (monthly count)', secondary_y=False)
fig.update_yaxes(title_text='Control events (monthly count)', secondary_y=True)
fig.show()

In [19]:
battles = monthly['Battles']
control = monthly['Control_signals']

lags = range(-6, 7)  # battles 6 months before to 6 months after control changes
correlations = []
for lag in lags:
    if lag >= 0:
        # Battles leading control by `lag` months
        corr = battles.shift(lag).corr(control)
    else:
        corr = battles.corr(control.shift(-lag))
    correlations.append(corr)

lag_df = pd.DataFrame({'lag_months': list(lags), 'correlation': correlations})

fig = go.Figure(go.Bar(
    x=lag_df['lag_months'], y=lag_df['correlation'],
    marker_color=['steelblue' if l >= 0 else 'lightgray' for l in lag_df['lag_months']]
))
fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(
    title=f'{country}: Cross-correlation of battles and control events',
    xaxis_title='Lag (months) — positive = battles lead control changes',
    yaxis_title='Correlation',
    width=900, height=400
)
fig.show()

print(lag_df.to_string(index=False))

 lag_months  correlation
         -6     0.438967
         -5     0.460469
         -4     0.483559
         -3     0.483601
         -2     0.513428
         -1     0.526235
          0     0.560712
          1     0.528381
          2     0.555622
          3     0.545633
          4     0.583561
          5     0.544815
          6     0.544211


## Actors and Interactions

In [24]:
import pandas as pd

# ---------- 1. All unique actor1 values ----------
print("="*70)
print(f"ACTOR1 — {df['actor1'].nunique()} unique values")
print("="*70)
actor1_counts = df['actor1'].value_counts(dropna=False)
print(actor1_counts.to_string())

# ---------- 2. All unique actor2 values ----------
print("\n" + "="*70)
print(f"ACTOR2 — {df['actor2'].nunique()} unique values")
print("="*70)
actor2_counts = df['actor2'].value_counts(dropna=False)
print(actor2_counts.to_string())

# ---------- 3. All unique interaction codes ----------
print("\n" + "="*70)
print(f"INTERACTION — {df['interaction'].nunique()} unique values")
print("="*70)
interaction_counts = df['interaction'].value_counts(dropna=False).sort_index()
print(interaction_counts.to_string())

# ---------- 4. inter1 and inter2 individually (the actor-type codes) ----------
if 'inter1' in df.columns and 'inter2' in df.columns:
    print("\n" + "="*70)
    print("INTER1 (actor1 type) and INTER2 (actor2 type)")
    print("="*70)
    print("\ninter1:")
    print(df['inter1'].value_counts(dropna=False).sort_index().to_string())
    print("\ninter2:")
    print(df['inter2'].value_counts(dropna=False).sort_index().to_string())

ACTOR1 — 2050 unique values
actor1
Military Forces of Myanmar (2021-)                                                                                                                                                                                        42827
Protesters (Myanmar)                                                                                                                                                                                                      18163
Unidentified Armed Group (Myanmar)                                                                                                                                                                                         6990
Military Forces of Myanmar (2016-2021)                                                                                                                                                                                     3617
Civilians (Myanmar)                                                  